# Simulating Data From Bayesian Networks

pgmpy implements the `DiscreteBayesianNetwork.simulate` method to allow users to simulate data from a fully defined Bayesian Network under various conditions. These conditions can be any combination of:
1. Virtual Evidence
2. Hard Evidence
3. Virtual Intervention
4. Hard Intervention

Users can also provide data corresponding to some of the variables in the model (partial samples) to the simulation method. This allows users to fix the values of those variables to the specified value.

Lastly, the user can also generate data with missing values, according to a user-defined CPD, to simulate realistic real-world data and evaluate how missingness affects inference.

In [132]:
# A helper function to compute probability distributions from simulated samples.
def get_distribution(samples, variables=None):
    """
    For marginal distribution, P(A): get_distribution(samples, variables=['A'])
    For joint distribution, P(A, B): get_distribution(samples, variables=['A', 'B'])
    """
    if variables is None:
        raise ValueError("variables must be specified")

    return samples.groupby(variables).size() / samples.shape[0]

In [133]:
# Do not print warnings
import logging
from pgmpy.global_vars import logger
logger.setLevel(logging.ERROR)

# Specify the model to simulate data from.
from pgmpy.factors.discrete import TabularCPD
from pgmpy.utils import get_example_model

import numpy as np
import pandas as pd

alarm = get_example_model("alarm")

## 1. Standard simulation

Without any specified conditions for simulation, the `simulate` method draws samples from the joint distribution of the model.

In [134]:
samples = alarm.simulate(n_samples=int(1e4))
samples.head()

Generating for node: BP: 100%|██████████| 37/37 [00:00<00:00, 123.37it/s]      


,PCWP,CVP,HRBP,PAP,CO,MINVOLSET,PRESS,INSUFFANESTH,CATECHOL,SAO2,...,LVEDVOLUME,KINKEDTUBE,HYPOVOLEMIA,HISTORY,VENTTUBE,STROKEVOLUME,EXPCO2,SHUNT,LVFAILURE,HR
0,NORMAL,NORMAL,LOW,NORMAL,LOW,NORMAL,HIGH,FALSE,NORMAL,LOW,...,NORMAL,FALSE,FALSE,FALSE,LOW,LOW,LOW,NORMAL,FALSE,NORMAL
1,LOW,LOW,HIGH,NORMAL,LOW,NORMAL,LOW,FALSE,HIGH,LOW,...,LOW,FALSE,TRUE,TRUE,LOW,LOW,LOW,NORMAL,TRUE,HIGH
2,NORMAL,LOW,HIGH,NORMAL,HIGH,NORMAL,NORMAL,FALSE,HIGH,LOW,...,NORMAL,FALSE,FALSE,FALSE,LOW,NORMAL,LOW,NORMAL,FALSE,HIGH
3,NORMAL,NORMAL,NORMAL,NORMAL,HIGH,NORMAL,HIGH,FALSE,HIGH,LOW,...,NORMAL,FALSE,FALSE,FALSE,ZERO,HIGH,LOW,NORMAL,FALSE,HIGH
4,NORMAL,NORMAL,HIGH,NORMAL,HIGH,NORMAL,LOW,FALSE,HIGH,LOW,...,NORMAL,TRUE,FALSE,FALSE,LOW,NORMAL,LOW,NORMAL,FALSE,HIGH


## 2. Simulation under specified evidence

Specifying hard evidence for some variables fixes their values to the specified evidence value during simulation.

In [135]:
evidence = {"CVP": "NORMAL", "HR": "HIGH"}
samples = alarm.simulate(n_samples=int(1e4), evidence=evidence)
samples.head()

100%|██████████| 10000/10000 [00:00<00:00, 23895.41it/s]


,PCWP,CVP,HRBP,PAP,CO,MINVOLSET,PRESS,INSUFFANESTH,CATECHOL,SAO2,...,LVEDVOLUME,KINKEDTUBE,HYPOVOLEMIA,HISTORY,VENTTUBE,STROKEVOLUME,EXPCO2,SHUNT,LVFAILURE,HR
0,NORMAL,NORMAL,HIGH,NORMAL,HIGH,NORMAL,NORMAL,FALSE,HIGH,HIGH,...,NORMAL,FALSE,FALSE,FALSE,LOW,NORMAL,LOW,NORMAL,FALSE,HIGH
1,NORMAL,NORMAL,HIGH,NORMAL,HIGH,NORMAL,LOW,FALSE,HIGH,LOW,...,NORMAL,FALSE,FALSE,FALSE,LOW,NORMAL,LOW,NORMAL,FALSE,HIGH
2,NORMAL,NORMAL,HIGH,LOW,HIGH,NORMAL,LOW,FALSE,HIGH,LOW,...,NORMAL,FALSE,FALSE,FALSE,ZERO,NORMAL,LOW,HIGH,FALSE,HIGH
3,NORMAL,NORMAL,HIGH,NORMAL,HIGH,NORMAL,LOW,FALSE,HIGH,LOW,...,NORMAL,FALSE,FALSE,FALSE,LOW,NORMAL,LOW,NORMAL,FALSE,HIGH
4,NORMAL,NORMAL,HIGH,NORMAL,HIGH,NORMAL,LOW,FALSE,HIGH,LOW,...,NORMAL,TRUE,FALSE,FALSE,LOW,NORMAL,LOW,NORMAL,FALSE,HIGH


In [136]:
# All values of HR and CVP should be set to HIGH and NORMAL respectively.
print(all(samples.HR == "HIGH"))
print(all(samples.CVP == "NORMAL"))

True
True


## 3. Simulation under soft/virtual evidence

Unlike hard evidence where the value of the specified variables is fixed to the specified evidence, virtual evidence allows users to set the marginal distribution of variables.

In [137]:
# The virtual evidence is specified using TabularCPDs. Here, P(CVP=NORMAL) = 0.2, P(CVP=LOW) = 0.3, and P(CPV=HIGH) = 0.5
cvp_evidence = TabularCPD(variable="CVP",
                          variable_card=3,
                          values=[[0.2], [0.3], [0.5]],
                          state_names={"CVP": ["LOW", "NORMAL", "HIGH"]})
samples = alarm.simulate(n_samples=int(1e4), virtual_evidence=[cvp_evidence])

100%|██████████| 10000/10000 [00:01<00:00, 8777.99it/s]


In [138]:
# Check the marginal distribution of CVP
get_distribution(samples, variables=['CVP'])

/tmp/ipykernel_71984/3805155865.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return samples.groupby(variables).size() / samples.shape[0]


CVP
HIGH      0.2360
LOW       0.0771
NORMAL    0.6869
dtype: float64

## 4. Simulation under specified intervention

Using the `do` argument, users can specify interventions to the model. The value of the intervened variables are set to the specified value and all incoming edges to these variables are removed in the model.

In [139]:
samples = alarm.simulate(n_samples=int(1e4), do={"CVP": "NORMAL", "HR": "HIGH"})
samples.head()

100%|██████████| 10000/10000 [00:01<00:00, 5082.66it/s]


,PCWP,CVP,HRBP,PAP,CO,MINVOLSET,PRESS,INSUFFANESTH,CATECHOL,SAO2,...,LVEDVOLUME,KINKEDTUBE,HYPOVOLEMIA,HISTORY,VENTTUBE,STROKEVOLUME,EXPCO2,SHUNT,LVFAILURE,HR
0,NORMAL,NORMAL,HIGH,NORMAL,HIGH,HIGH,HIGH,FALSE,HIGH,LOW,...,NORMAL,FALSE,FALSE,FALSE,LOW,NORMAL,LOW,NORMAL,FALSE,HIGH
1,HIGH,NORMAL,HIGH,NORMAL,HIGH,NORMAL,LOW,FALSE,HIGH,LOW,...,HIGH,TRUE,TRUE,FALSE,LOW,NORMAL,LOW,NORMAL,FALSE,HIGH
2,NORMAL,NORMAL,HIGH,NORMAL,LOW,NORMAL,HIGH,FALSE,HIGH,LOW,...,NORMAL,FALSE,FALSE,FALSE,LOW,NORMAL,LOW,NORMAL,FALSE,HIGH
3,NORMAL,NORMAL,HIGH,NORMAL,HIGH,NORMAL,NORMAL,FALSE,HIGH,LOW,...,NORMAL,FALSE,FALSE,FALSE,LOW,NORMAL,LOW,NORMAL,FALSE,HIGH
4,NORMAL,NORMAL,HIGH,LOW,HIGH,NORMAL,LOW,FALSE,HIGH,LOW,...,NORMAL,FALSE,FALSE,FALSE,LOW,NORMAL,LOW,NORMAL,FALSE,HIGH


## 5. Simulation under soft/virtual intervention

Similar to virtual evidence, users can specify virtual intervention as well.

In [140]:
cvp_intervention = TabularCPD(variable="CVP",
                              variable_card=3,
                              values=[[0.2], [0.3], [0.5]],
                              state_names={"CVP": ["LOW", "NORMAL", "HIGH"]})
samples = alarm.simulate(n_samples=int(1e4), virtual_intervention=[cvp_intervention])
get_distribution(samples, variables=["CVP"])  # P(HISTORY)

100%|██████████| 10000/10000 [00:01<00:00, 9451.92it/s]
/tmp/ipykernel_71984/3805155865.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return samples.groupby(variables).size() / samples.shape[0]


CVP
HIGH      0.3772
LOW       0.2117
NORMAL    0.4111
dtype: float64

## 6. Partial samples

Users can also pass already generated data for some variables (for example, from some other simulation) to the simulation. This is equivalent to separately specifying evidence for each sample that is generate.

In [141]:
# Generate some data on CVP.
partial_cvp = pd.DataFrame(np.random.choice(["LOW", "NORMAL", "HIGH"], int(1e4)), columns=['CVP'])
samples = alarm.simulate(n_samples=int(1e4), partial_samples=partial_cvp)

Generating for node: BP: 100%|██████████| 37/37 [00:00<00:00, 125.36it/s]      


In [142]:
print(all(samples["CVP"] == partial_cvp["CVP"]))

True


## 7. Simulate Missing data

Lastly, users can generate data with missing values for some specified variables, according to a user defined CPD. The name of the missing variable should be followed by a * to indicate missingness, and should contain 2 states: 1 (Missing) and 0 (Not Missing). Optionally, we can use the `return_full` argument to get back the removed values for comparison.

#### 7.1. Missing Completely At Random (MCAR)

In [143]:
# CVP data missing completely randomly with 0.4 probability
missing_CVP = TabularCPD(
    variable="CVP*",
    variable_card=2,
    values=[[0.6], 
            [0.4]], # Missing probability = 0.4
    state_names={"CVP*": [0, 1]}
)

samples = alarm.simulate(n_samples=1000, missing_prob=[missing_CVP], return_full=True)
samples.head()

Generating for node: BP: 100%|██████████| 38/38 [00:00<00:00, 380.61it/s]


,PCWP,CVP,HRBP,PAP,CO,MINVOLSET,PRESS,INSUFFANESTH,CATECHOL,SAO2,...,LVEDVOLUME,KINKEDTUBE,HYPOVOLEMIA,HISTORY,VENTTUBE,STROKEVOLUME,EXPCO2,SHUNT,LVFAILURE,HR
0,NORMAL,NaN,HIGH,NORMAL,LOW,NORMAL,LOW,FALSE,HIGH,LOW,...,HIGH,FALSE,TRUE,FALSE,LOW,LOW,LOW,HIGH,FALSE,HIGH
1,LOW,NaN,LOW,NORMAL,LOW,NORMAL,HIGH,FALSE,NORMAL,HIGH,...,NORMAL,FALSE,TRUE,FALSE,ZERO,LOW,LOW,NORMAL,FALSE,NORMAL
2,HIGH,HIGH,LOW,NORMAL,NORMAL,NORMAL,ZERO,FALSE,HIGH,LOW,...,HIGH,FALSE,FALSE,FALSE,ZERO,NORMAL,LOW,NORMAL,FALSE,NORMAL
3,NORMAL,NORMAL,HIGH,NORMAL,HIGH,NORMAL,HIGH,FALSE,HIGH,HIGH,...,NORMAL,FALSE,FALSE,FALSE,ZERO,NORMAL,LOW,NORMAL,FALSE,HIGH
4,HIGH,NORMAL,LOW,NORMAL,NORMAL,NORMAL,NORMAL,FALSE,HIGH,LOW,...,NORMAL,FALSE,FALSE,FALSE,LOW,NORMAL,LOW,NORMAL,FALSE,NORMAL


In [ ]:
print(f"Missing values: {samples["CVP"].isna().sum()}/{len(samples["CVP"])}")
print()

print("Original Distribution:")
print(get_distribution(samples, variables="CVP_full"))
print()
print("Distribution of Missing/Removed")
print(get_distribution(samples.loc[samples["CVP"].isna()], variables="CVP_full")) # Since removal was completely random, we expect minimal change in distribution

Missing values: 400/1000

Original Distribution:
CVP_full
HIGH      0.161
LOW       0.121
NORMAL    0.718
dtype: float64

Distribution of Missing/Removed
CVP_full
HIGH      0.1725
LOW       0.1150
NORMAL    0.7125
dtype: float64


/tmp/ipykernel_71984/3805155865.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return samples.groupby(variables).size() / samples.shape[0]
/tmp/ipykernel_71984/3805155865.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return samples.groupby(variables).size() / samples.shape[0]


#### 7.2. Missing At Random (MAR)

In [ ]:
# CVP data missing depending on the observed LVEDVOLUME
missing_CVP = TabularCPD(
    variable="CVP*",
    variable_card=2,
    values=[[0.8, 0.2, 0.7], 
            [0.2, 0.8, 0.3]], # Missing probabilities: LOW = 0.2, NORMAL = 0.8, HIGH = 0.3
    evidence=["LVEDVOLUME"],
    evidence_card=[3],
    state_names={
        "CVP*": [0, 1],
        "LVEDVOLUME": ["LOW", "NORMAL", "HIGH"]}
)

samples = alarm.simulate(n_samples=1000, missing_prob=[missing_CVP], return_full=True)
samples.head()

Generating for node: BP: 100%|██████████| 38/38 [00:00<00:00, 355.48it/s]


,PCWP,CVP,HRBP,PAP,CO,MINVOLSET,PRESS,INSUFFANESTH,CATECHOL,SAO2,...,LVEDVOLUME,KINKEDTUBE,HYPOVOLEMIA,HISTORY,VENTTUBE,STROKEVOLUME,EXPCO2,SHUNT,LVFAILURE,HR
0,NORMAL,NaN,HIGH,NORMAL,HIGH,NORMAL,HIGH,FALSE,HIGH,LOW,...,NORMAL,FALSE,FALSE,FALSE,ZERO,NORMAL,ZERO,NORMAL,FALSE,HIGH
1,HIGH,HIGH,HIGH,NORMAL,HIGH,LOW,HIGH,FALSE,HIGH,LOW,...,HIGH,FALSE,FALSE,FALSE,ZERO,HIGH,LOW,NORMAL,FALSE,HIGH
2,HIGH,NaN,HIGH,NORMAL,HIGH,NORMAL,NORMAL,TRUE,HIGH,LOW,...,HIGH,FALSE,TRUE,FALSE,LOW,NORMAL,LOW,NORMAL,FALSE,HIGH
3,NORMAL,NaN,NORMAL,NORMAL,HIGH,NORMAL,LOW,FALSE,HIGH,LOW,...,NORMAL,FALSE,FALSE,FALSE,LOW,NORMAL,LOW,NORMAL,FALSE,HIGH
4,HIGH,NaN,HIGH,NORMAL,HIGH,NORMAL,HIGH,FALSE,HIGH,LOW,...,HIGH,FALSE,TRUE,FALSE,LOW,NORMAL,HIGH,NORMAL,FALSE,HIGH


In [ ]:
print(f"Missing values: {samples["CVP"].isna().sum()}/{len(samples["CVP"])}")
print()

print("Original Distribution:")
print(get_distribution(samples, variables=["LVEDVOLUME", "CVP_full"]))
print()
print("Distribution of Missing/Removed")
print(get_distribution(samples.loc[samples["CVP"].isna()], variables=["LVEDVOLUME", "CVP_full"])) # Since probability of missing is higher when LVEDVOLUME is "NORMAL" we expect distribution to be higher values there, and lesser otherwise

Missing values: 626/1000

Original Distribution:
LVEDVOLUME  CVP_full
HIGH        HIGH        0.150
            LOW         0.000
            NORMAL      0.064
LOW         HIGH        0.001
            LOW         0.086
            NORMAL      0.002
NORMAL      HIGH        0.008
            LOW         0.024
            NORMAL      0.665
dtype: float64

Distribution of Missing/Removed
LVEDVOLUME  CVP_full
HIGH        HIGH        0.068690
            LOW         0.000000
            NORMAL      0.022364
LOW         HIGH        0.000000
            LOW         0.028754
            NORMAL      0.000000
NORMAL      HIGH        0.009585
            LOW         0.033546
            NORMAL      0.837061
dtype: float64


/tmp/ipykernel_71984/3805155865.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return samples.groupby(variables).size() / samples.shape[0]
/tmp/ipykernel_71984/3805155865.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return samples.groupby(variables).size() / samples.shape[0]


#### Missing Not At Random (MNAR)

In [ ]:
# CVP data missing depending on the unobserved original CVP value
missing_CVP = TabularCPD(
    variable="CVP*",
    variable_card=2,
    values=[[0.2, 0.4, 0.6], 
            [0.8, 0.6, 0.4]], # Missing probabilities: LOW = 0.8, NORMAL = 0.6, HIGH = 0.4
    evidence=["CVP"],
    evidence_card=[3],
    state_names={
        "CVP*": [0, 1],
        "CVP": ["LOW", "NORMAL", "HIGH"]}
)

samples = alarm.simulate(n_samples=1000, missing_prob=[missing_CVP], return_full=True)
samples.head()

Generating for node: BP: 100%|██████████| 38/38 [00:00<00:00, 291.39it/s]      


,PCWP,CVP,HRBP,PAP,CO,MINVOLSET,PRESS,INSUFFANESTH,CATECHOL,SAO2,...,LVEDVOLUME,KINKEDTUBE,HYPOVOLEMIA,HISTORY,VENTTUBE,STROKEVOLUME,EXPCO2,SHUNT,LVFAILURE,HR
0,NORMAL,NaN,NORMAL,HIGH,HIGH,NORMAL,HIGH,FALSE,HIGH,LOW,...,LOW,TRUE,TRUE,TRUE,LOW,LOW,LOW,NORMAL,TRUE,HIGH
1,NORMAL,NaN,HIGH,NORMAL,HIGH,HIGH,HIGH,FALSE,HIGH,HIGH,...,NORMAL,FALSE,FALSE,FALSE,ZERO,NORMAL,LOW,NORMAL,FALSE,HIGH
2,NORMAL,NORMAL,LOW,NORMAL,NORMAL,NORMAL,HIGH,FALSE,NORMAL,LOW,...,NORMAL,FALSE,FALSE,FALSE,LOW,NORMAL,LOW,NORMAL,FALSE,NORMAL
3,NORMAL,NORMAL,LOW,NORMAL,NORMAL,NORMAL,NORMAL,FALSE,HIGH,LOW,...,NORMAL,FALSE,FALSE,FALSE,LOW,NORMAL,LOW,NORMAL,FALSE,NORMAL
4,HIGH,NaN,HIGH,NORMAL,HIGH,NORMAL,NORMAL,FALSE,HIGH,LOW,...,HIGH,FALSE,FALSE,FALSE,LOW,NORMAL,LOW,NORMAL,FALSE,HIGH


In [ ]:
print(f"Missing values: {samples["CVP"].isna().sum()}/{len(samples["CVP"])}")
print()

print("Original Distribution:")
print(get_distribution(samples, variables="CVP_full"))
print()
print("Distribution of Missing/Removed")
print(get_distribution(samples.loc[samples["CVP"].isna()], variables="CVP_full")) # Since probability of missing is higher when CVP is "LOW" and lower when "CVP" is high we expect missing distribution for "LOW" to be greater, and for "HIGH" to be lower

Missing values: 600/1000

Original Distribution:
CVP_full
HIGH      0.143
LOW       0.125
NORMAL    0.732
dtype: float64

Distribution of Missing/Removed
CVP_full
HIGH      0.100000
LOW       0.171667
NORMAL    0.728333
dtype: float64


/tmp/ipykernel_71984/3805155865.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return samples.groupby(variables).size() / samples.shape[0]
/tmp/ipykernel_71984/3805155865.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return samples.groupby(variables).size() / samples.shape[0]
